# 02 — Benchmark: 4-arm GraphRAG vs PlainRAG ablation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vardhjain/graphrag-pubmedqa-ablation/blob/main/notebooks/02_benchmark.ipynb)

Thin Colab wrapper around `scripts/run_benchmark.py` and `scripts/compare.py`.

**Use a GPU runtime** (a faster GPU mainly cuts wall-clock since this is
LLM-inference-bound). Add `ARANGO_PASS` and `ARANGO_HOST` in the Colab **Secrets**
panel. Run `01_ingest.ipynb` first.

Arms (each isolates one component):

| arm | adds |
| --- | --- |
| `plain` | vector top-k chunks (baseline) |
| `plain_rr` | + cross-encoder reranker |
| `graph` | + parent-paper expansion (full abstracts) |
| `graph_concepts` | + MeSH concept-hop expansion |
| `graph_norr` | + parent-paper expansion, no reranker (matches the hosted demo's config) |

The runner retries failed questions and auto-restarts Ollama if it crashes, and it
checkpoints every 25 questions — so a transient error can't abort an arm.

In [ ]:
# Confirm the GPU (optional).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Reset-safe clone: always starts from /content and removes any prior copy,
# so re-running this cell can never nest a second checkout.
%cd /content
!rm -rf graphrag-pubmedqa-ablation
!git clone -b main https://github.com/vardhjain/graphrag-pubmedqa-ablation.git -q
%cd graphrag-pubmedqa-ablation
!pip install -q -r requirements.txt

In [ ]:
# Install Ollama and pull the LLM (once). The benchmark script manages the
# server from here on (health-check + auto-restart).
!which ollama || (apt-get install -y zstd -q && curl -fsSL https://ollama.com/install.sh | sh)
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama pull deepseek-r1:8b

In [ ]:
import os
from google.colab import userdata

# Connection settings from Colab Secrets (nothing hardcoded).
for key in ['ARANGO_PASS', 'ARANGO_HOST', 'ARANGO_DB']:
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
    except Exception:
        pass
assert os.environ.get('ARANGO_PASS'), 'Add ARANGO_PASS in the Secrets panel.'

# Generation knobs (identical across arms, so the comparison is unaffected).
# Raise NUM_CTX on a large-VRAM GPU; lower it on a small one if you hit OOM.
os.environ['LLM_NUM_CTX'] = '8192'
os.environ['LLM_NUM_PREDICT'] = '1024'
print('ARANGO_HOST:', os.environ.get('ARANGO_HOST', '(default http://localhost:8529)'))

In [ ]:
# Run all five arms. The chunk corpus is downloaded once and cached, then
# reused by every arm (identical corpus -> fair comparison). Each arm saves its
# own results JSON, so if one dies you can re-run just that arm.
for arm in ['plain', 'plain_rr', 'graph', 'graph_concepts', 'graph_norr']:
    print(f'\n===== {arm} =====')
    !python scripts/run_benchmark.py --arm {arm} --n 200

In [ ]:
# Summary table, paired McNemar tests, and the ablation figure.
!python scripts/compare.py
from IPython.display import Image, display, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('results/ablation.png'))

In [ ]:
# Optional: commit results back to GitHub (set a PAT first).
# !git config user.email you@example.com && git config user.name you
# !git add results/ && git commit -m 'Add benchmark results' && git push